In [1]:
# Licensed to the Apache Software Foundation (ASF) under one
# or more contributor license agreements.  See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership.  The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License.  You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing,
# software distributed under the License is distributed on an
# "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied.  See the License for the
# specific language governing permissions and limitations
# under the License.

In [ ]:
!pip install apache-burr

# Stopping a streaming action early

When you stop reading a stream before it ends, you want the action's cleanup to run *now*, the step to be reported as failed (so resuming re-runs it), and the state not to be updated. Each demo below prints a timeline of what happened, in order. The code is in [application.py](application.py), and the explanation is in [README.md](README.md).

In [3]:
import asyncio

import application as demo

## Pattern 1: `break`, then `container.close()` / `await container.aclose()`

In [4]:
demo.stop_with_close_demo()


=== Pattern 1: break, then container.close()
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   post_run_step: respond failed with GeneratorExit()
   respond: close LLM connection
   caller: closed the container
   caller: get() -> result=None, 'response' in state=False


In [5]:
await demo.stop_with_aclose_demo()


=== Pattern 1 (async): break, then await container.aclose()
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   respond: close LLM connection
   post_run_step: respond failed with GeneratorExit()
   caller: closed the container


## Pattern 2: `contextlib.closing(app.stream_iterate(...))`

However you leave the block, the in-flight action is closed.

In [6]:
demo.stop_iterate_with_closing_demo()


=== Pattern 2: contextlib.closing(app.stream_iterate(...))
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   post_run_step: respond failed with GeneratorExit()
   respond: close LLM connection
   caller: left the `with` block


## Pattern 3: a client disconnect cancels the consuming task

The hook receives `CancelledError`. See [server.py](server.py) for the FastAPI version.

In [7]:
await demo.client_disconnect_demo()


=== Pattern 3: client disconnect, cancel the consuming task
   respond: open LLM connection
   server: sent 'Burr '
   server: sent 'streams '
   server: client disconnected -> task.cancel()
   respond: close LLM connection
   post_run_step: respond failed with CancelledError()


## Persistence: a stopped step is saved as `failed`, and resuming re-runs it

In [8]:
demo.resume_after_stop_demo()


=== Persistence: a stopped step is 'failed' and is re-run on resume
   respond: open LLM connection
   post_run_step: respond failed with GeneratorExit()
   respond: close LLM connection
   persister: last step saved as 'failed'
   resume: next action is 'respond' (it runs again)


## Cleanup errors are raised by `close()`

In [9]:
demo.cleanup_error_demo()


=== Cleanup errors: close() raises them to the caller
   respond: open LLM connection
   caller: got 'Burr '
   caller: got 'streams '
   caller: got 'tokens '
   post_run_step: respond failed with GeneratorExit()
   respond: close LLM connection
   caller: close() raised ConnectionError('LLM connection did not close cleanly')
